# 맥락 조건부 LoRA 가중치 조향 — 재현용 실험 노트북

**개인 사전 실험 | 이석규 | 2026년 9월**

서로 다른 응답 스타일(공감형·직언형)을 학습한 LoRA 두 개를 동일한 기반 모델에서 독립적으로 학습하고, 가중치 변화량의 조합 비율에 따라 출력이 어떻게 달라지는지 확인합니다.

- 기반 모델: `Qwen/Qwen2.5-1.5B-Instruct`
- 실험 환경: Google Colab의 NVIDIA T4 GPU, FP16
- 학습: 각 12개 예시, 3 epoch, 36 step, `r=8`, `lora_alpha=16`, `q_proj`·`v_proj`
- 조합: `PEFT.add_weighted_adapter(..., combination_type="cat")`, 공감형 비율 0/25/50/75/100%
- 평가: 학습에 쓰지 않은 질문 4개에 대한 탐색적 응답 비교

**중요:** 이 노트북은 기존 Colab 작업 기록에서 재연결·오류 복구·중복 코드를 제거해 **실행 순서가 보이도록 재구성한 사본**입니다. 아래 코드 셀의 출력은 삭제되어 있으며, **정리된 노트북을 처음부터 끝까지 재실행했다고 주장하지 않습니다.** 당시 실제 실행 결과는 GitHub 저장소의 [`mixing_ratio_results.json`](https://github.com/mongkoon/lora-steering-experiment/blob/main/results/mixing_ratio_results.json)과 [`multiple_context_results.json`](https://github.com/mongkoon/lora-steering-experiment/blob/main/results/multiple_context_results.json)에 있습니다. 재실행 시 라이브러리 버전, 초기화, GPU 환경에 따라 출력이 달라질 수 있습니다.

이 코드는 연구과제 전체의 재현, 맥락 조건부 매핑 모델 학습, 계층 선택, 실제 온디바이스 성능 검증을 포함하지 않습니다. **동일한 데이터 12개를 여러 번 학습했으며, 공감/직언 특성에 대한 정량적인 효과 검증도 아직 수행하지 않았습니다.**

**실행 순서:** 새 Colab 노트북에서 `GPU` 런타임을 선택한 뒤 아래 셀을 위에서 아래로 **각각 한 번만** 실행합니다. 원본 Colab에서 이미 학습한 어댑터를 재학습할 필요는 없으며, 이 노트북은 제출용 코드 확인 및 추후 재현용입니다.

**보관 주의:** 이 노트북의 학습·결과 저장 셀을 재실행하면 기존 Google Drive의 동일 경로 파일을 덮어쓸 수 있습니다. 기존 실험 결과를 보존하려면 별도 사본을 만든 뒤 실행하세요.


## 1. 라이브러리 준비

원본 실행 당시 PEFT와 Colab의 `torchao 0.10.0`이 충돌했습니다. 이번 실험에서는 torchao가 필요하지 않으므로 설치되어 있으면 제거합니다. 아래 설치 명령은 당시 동작한 주요 라이브러리 버전을 명시합니다. 현재 Colab에 설치된 PyTorch와 호환되지 않으면 추가 조정이 필요할 수 있습니다.

**설치 후 런타임 재시작 안내가 나타나면:** 런타임을 다시 시작하고, 새 세션에서 이 설치 셀의 실행을 건너뛴 채 다음 셀부터 실행하세요. 이미 PEFT/torchao를 import한 세션에서도 재시작이 필요할 수 있습니다.


In [ ]:
%pip uninstall -y torchao
%pip install -q "transformers==5.17.0" "peft==0.21.0" "accelerate==1.15.0" "datasets==5.0.1" "safetensors==0.8.0"


In [ ]:
import importlib.metadata as metadata
import torch
from pathlib import Path
from google.colab import drive

assert torch.cuda.is_available(), "Colab에서 GPU 런타임을 선택해 주세요."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
for pkg in ("transformers", "peft", "accelerate", "datasets", "safetensors"):
    print(pkg, metadata.version(pkg))
try:
    print("torchao:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("torchao: not installed")

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/lora-steering-experiment")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


## 2. 기반 모델 불러오기

초기 탐색에서 0.5B 모델과 한국어 응답을 테스트했으나, 최종 비교 실험은 **1.5B 모델·영어 입력**으로 진행했습니다. 여기서는 최종 실험에 사용한 모델만 불러옵니다.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 실험에 사용할 기반 언어모델
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 기본 언어모델 불러오기
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("=== Model Information ===")
print("Model:", MODEL_ID)
print("Parameters:", f"{model.num_parameters():,}")
print("Device:", next(model.parameters()).device)
print("Data type:", next(model.parameters()).dtype)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nModel loaded successfully!")


## 3. 학습 데이터 생성

동일한 사용자 질문 12개에 대한 공감형·직언형 답변을 쌍으로 구성합니다. 아래 데이터 생성 코드는 **원본 노트북의 학습 데이터 생성 셀**에서 가져왔습니다. 질문과 답변은 본 소규모 실험을 위해 준비한 예시이며, 외부 평가셋으로 검증되지 않았습니다.


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("/content/lora_steering_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 동일한 질문에 대한 공감형 / 직언형 응답
# 각 항목: (사용자 질문, 공감형 응답, 직언형 응답)

training_pairs = [
    (
        "I missed an important deadline at work. What should I do?",
        "Missing an important deadline can feel stressful. "
        "It is understandable to feel worried about the consequences. "
        "Give yourself a moment to regroup, then explain the situation "
        "to your manager and discuss a realistic recovery plan.",

        "Inform your manager immediately. Explain why the deadline "
        "was missed, provide a revised completion date, and identify "
        "the steps needed to prevent the same problem."
    ),

    (
        "I received negative feedback on my project. How should I respond?",
        "Receiving criticism on something you worked hard on can be "
        "discouraging. Your effort still matters, even when the result "
        "needs improvement. Take time to understand the feedback and "
        "decide which changes would be most helpful.",

        "Review the feedback and separate actionable points from "
        "general opinions. Identify the three most important changes, "
        "revise your project, and request another review."
    ),

    (
        "I failed an important exam and feel disappointed.",
        "Failing an important exam can be painful, especially when "
        "you invested a lot of effort. One result does not define "
        "your ability. Allow yourself some time to recover before "
        "reviewing your preparation and planning your next attempt.",

        "Review your exam results to identify weak areas. Create "
        "a study schedule, practice with previous exam questions, "
        "and track your progress before the next attempt."
    ),

    (
        "I had an argument with a close friend. What should I do?",
        "Arguments with close friends can be emotionally difficult. "
        "It makes sense that you want to repair the relationship. "
        "Consider giving both of you some time to settle down "
        "before reaching out for an honest conversation.",

        "Identify the cause of the disagreement. Contact your friend, "
        "acknowledge your part in the conflict, and discuss how "
        "to avoid repeating the same problem."
    ),

    (
        "I made a mistake while presenting to my team.",
        "Making a mistake in front of colleagues can feel embarrassing. "
        "Most people experience moments like this, and one mistake "
        "does not erase your preparation or ability. Reflect on "
        "what happened and use it to prepare for your next presentation.",

        "Identify exactly what went wrong during the presentation. "
        "Correct any inaccurate information, practice the difficult "
        "sections, and prepare a checklist for your next presentation."
    ),

    (
        "I am overwhelmed by the number of tasks I need to finish.",
        "Having too many responsibilities at once can be exhausting. "
        "It is understandable to feel overwhelmed when everything "
        "seems urgent. Give yourself permission to focus on one "
        "manageable step at a time and ask for support if needed.",

        "List every outstanding task. Rank them by urgency and "
        "importance, estimate the time required, and schedule "
        "the highest-priority tasks first. Delegate or postpone "
        "lower-priority work."
    ),

    (
        "My manager rejected an idea I worked hard on.",
        "It can be disappointing when an idea you care about is "
        "rejected. Your effort and creativity still have value. "
        "Consider asking your manager for feedback so you can "
        "understand their concerns and decide how to move forward.",

        "Ask your manager for specific reasons for the rejection. "
        "Evaluate the objections, revise your proposal where "
        "appropriate, and present a more focused version if justified."
    ),

    (
        "I am nervous about starting a new job.",
        "Starting a new job can bring both excitement and uncertainty. "
        "Feeling nervous is natural when entering an unfamiliar "
        "environment. Give yourself time to learn the role "
        "and remember that you do not need to know everything immediately.",

        "Review your job responsibilities before your first day. "
        "Learn the team's tools and processes, clarify expectations "
        "with your manager, and create a plan for your first month."
    ),

    (
        "A customer complained about my work. How should I handle it?",
        "Receiving a complaint can be uncomfortable, especially "
        "when you have tried your best. You can take the concern "
        "seriously without treating it as a judgment of your worth. "
        "Listen carefully and focus on understanding the customer's experience.",

        "Document the complaint, verify the underlying issue, "
        "and determine whether corrective action is needed. "
        "Communicate the resolution and confirm that the "
        "customer's concern has been addressed."
    ),

    (
        "I feel uncertain about changing my career.",
        "A career change can feel intimidating because it involves "
        "leaving familiar routines behind. It is reasonable to "
        "feel uncertain about such a major decision. Take time "
        "to explore your interests and consider what matters most to you.",

        "Define your target role, identify the required skills, "
        "and compare them with your current experience. "
        "Research job opportunities, create a learning plan, "
        "and test your interest through a small project."
    ),

    (
        "I received criticism from a colleague during a meeting.",
        "Being criticized in front of others can feel uncomfortable. "
        "It is understandable if you need time to process what "
        "happened. Consider whether the feedback contains useful "
        "information and whether a private conversation would help.",

        "Review the criticism for factual accuracy. Address any "
        "valid issues, request clarification where necessary, "
        "and discuss communication expectations privately "
        "with your colleague."
    ),

    (
        "I am afraid to speak in public because I might make mistakes.",
        "Public speaking can be intimidating, and worrying about "
        "mistakes is a common experience. You do not have to deliver "
        "a perfect speech to communicate effectively. Start with "
        "small practice sessions and build confidence gradually.",

        "Prepare a clear outline and rehearse your speech aloud. "
        "Record a practice session, identify weak sections, "
        "and repeat until you can deliver the main points "
        "without relying on a full script."
    ),
]

# 두 데이터셋의 시스템 메시지는 동일하게 유지
SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Respond to the user's question naturally."
)

empathetic_data = []
direct_data = []

for question, empathetic_answer, direct_answer in training_pairs:

    empathetic_data.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": empathetic_answer}
        ]
    })

    direct_data.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": direct_answer}
        ]
    })


# JSONL 형식으로 각각 저장
def save_jsonl(filename, records):
    filepath = DATA_DIR / filename

    with open(filepath, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    return filepath


empathetic_path = save_jsonl(
    "empathetic_train.jsonl",
    empathetic_data
)

direct_path = save_jsonl(
    "direct_train.jsonl",
    direct_data
)

# 저장 결과 확인
print("=== Dataset Created ===")
print("Empathetic samples:", len(empathetic_data))
print("Direct samples:", len(direct_data))

print("\n=== Saved Files ===")
print(empathetic_path)
print(direct_path)

print("\n=== Sample Question ===")
print(training_pairs[0][0])

print("\n=== Empathetic Answer ===")
print(training_pairs[0][1])

print("\n=== Direct Answer ===")
print(training_pairs[0][2])


In [ ]:
# Colab 임시 저장공간이 초기화되어도 학습 데이터를 다시 입력하지 않도록 Drive에 백업
import shutil
BACKUP_DATA_DIR = PROJECT_DIR / "data"
BACKUP_DATA_DIR.mkdir(parents=True, exist_ok=True)
for fname in ("empathetic_train.jsonl", "direct_train.jsonl"):
    shutil.copy2(DATA_DIR / fname, BACKUP_DATA_DIR / fname)
    print("백업:", BACKUP_DATA_DIR / fname)


## 4. 학습 입력 전처리

시스템 메시지와 사용자 질문 부분은 `labels=-100`으로 마스킹하여 손실 계산에서 제외하고, assistant 답변 토큰만 학습합니다. 모델 입력은 Qwen의 대화 템플릿을 사용합니다.


In [ ]:
import torch

# 최대 입력 길이
MAX_LENGTH = 160


def prepare_training_data(records):

    prepared = []

    for record in records:

        messages = record["messages"]

        # 시스템 메시지 + 사용자 질문
        # + assistant 응답 시작 부분
        prompt_text = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=False,
            add_generation_prompt=True
        )

        # assistant 정답까지 포함한 전체 대화
        full_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        # 특수 토큰 중복 삽입 방지
        prompt_ids = tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]

        full_ids = tokenizer(
            full_text,
            add_special_tokens=False
        )["input_ids"]

        # 토큰 단위에서도 질문 영역이
        # 전체 대화의 시작 부분과 일치하는지 확인
        assert full_ids[:len(prompt_ids)] == prompt_ids, (
            "Prompt token prefix mismatch"
        )

        # 데이터가 최대 길이를 초과하지 않는지 확인
        assert len(full_ids) <= MAX_LENGTH, (
            f"Sequence too long: {len(full_ids)}"
        )

        # 실제 모델 입력
        input_ids = full_ids

        # 질문 영역은 -100으로 설정하여 손실 계산 제외
        # assistant 응답 영역만 정답 토큰으로 사용
        labels = (
            [-100] * len(prompt_ids)
            + full_ids[len(prompt_ids):]
        )

        attention_mask = [1] * len(input_ids)

        assert len(input_ids) == len(labels)
        assert len(input_ids) == len(attention_mask)

        assert any(label != -100 for label in labels)

        prepared.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        })

    return prepared


# 공감형 / 직언형 학습 데이터 변환
empathetic_train = prepare_training_data(
    empathetic_data
)

direct_train = prepare_training_data(
    direct_data
)


# 결과 확인
print("=== Training Data Prepared ===")

print(
    "Empathetic samples:",
    len(empathetic_train)
)

print(
    "Direct samples:",
    len(direct_train)
)


# 첫 번째 공감형 학습 데이터 확인
sample = empathetic_train[0]

print("\n=== First Sample ===")

print(
    "Total tokens:",
    len(sample["input_ids"])
)

print(
    "Masked tokens:",
    sample["labels"].count(-100)
)

print(
    "Training tokens:",
    sum(
        label != -100
        for label in sample["labels"]
    )
)


# 실제 학습 대상인 assistant 응답만 복원
training_token_ids = [
    label
    for label in sample["labels"]
    if label != -100
]

print("\n=== Training Target ===")

print(
    tokenizer.decode(
        training_token_ids,
        skip_special_tokens=True
    )
)

print("\nTraining data preparation completed!")


## 5. 공감형 어댑터 초기화

원본과 같은 설정 `r=8`, `alpha=16`, `dropout=0`, `q_proj`·`v_proj`를 사용합니다. 기반 모델의 전체 가중치는 고정하고 LoRA만 학습합니다.


In [ ]:
import torch
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# 기본 모델 및 학습 데이터 확인
assert not isinstance(model, PeftModel), (
    "이미 LoRA가 적용된 모델입니다. "
    "이 셀을 중복 실행하지 마세요."
)

assert len(empathetic_train) == 12
assert len(direct_train) == 12
assert next(model.parameters()).device.type == "cuda"

# LoRA 설정
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 기본 모델에 공감형 LoRA 어댑터 추가
model = get_peft_model(
    model,
    lora_config,
    adapter_name="empathetic"
)

# 학습 가능한 파라미터 확인
print("=== LoRA Model ===")
print("Base model:", MODEL_ID)
print("Active adapter:", model.active_adapter)

print("\n=== Trainable Parameters ===")
model.print_trainable_parameters()

# LoRA가 적용된 모듈 확인
lora_modules = [
    name
    for name, module in model.named_modules()
    if hasattr(module, "lora_A")
]

print("\n=== LoRA Modules ===")
print("Total:", len(lora_modules))
print("First module:", lora_modules[0])

print("\n=== GPU Memory ===")
print(
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("\nEmpathetic LoRA adapter initialized successfully!")


## 6. 공감형 LoRA 학습 및 Drive 저장

이 셀은 실제로 어댑터의 가중치를 업데이트합니다. **한 번만 실행**하세요. 원본 실험에서는 3 epoch, 총 36 step을 수행했고, epoch 평균 loss는 2.4578 → 1.9735 → 1.7262였습니다. 이는 **당시 학습 실행 기록**이지 정리된 노트북의 재실행 결과가 아닙니다.


In [ ]:
import torch
import random
from pathlib import Path
from torch.optim import AdamW

# ========================================
# 1. 학습 환경 확인
# ========================================

assert model.active_adapter == "empathetic", (
    "공감형 LoRA가 활성화되어 있지 않습니다."
)

assert len(empathetic_train) == 12

# 재현성을 위한 난수 설정
random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# ========================================
# 2. 학습 설정
# ========================================

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4

# LoRA 파라미터만 학습
trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE
)

# FP16 학습의 수치적 안정성을 위한 GradScaler
scaler = torch.amp.GradScaler("cuda")

# 학습 모드 활성화
model.train()

# ========================================
# 3. 공감형 LoRA 학습
# ========================================

print("=== Empathetic LoRA Training ===")
print("Samples:", len(empathetic_train))
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Trainable parameters:", sum(
    p.numel() for p in trainable_params
))

print("\nTraining started...\n")

global_step = 0
epoch_losses = []

for epoch in range(NUM_EPOCHS):

    # 매 Epoch마다 데이터 순서 변경
    sample_indices = list(range(len(empathetic_train)))
    random.shuffle(sample_indices)

    total_loss = 0.0

    for idx in sample_indices:

        sample = empathetic_train[idx]

        # 학습 데이터를 GPU 텐서로 변환
        batch = {
            key: torch.tensor(
                [value],
                dtype=torch.long,
                device="cuda"
            )
            for key, value in sample.items()
        }

        optimizer.zero_grad(set_to_none=True)

        # FP16 혼합 정밀도 학습
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        # 비정상적인 loss 확인
        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Invalid loss at step {global_step + 1}: "
                f"{loss.item()}"
            )

        # 역전파
        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0
        )

        # LoRA 파라미터 업데이트
        scaler.step(optimizer)
        scaler.update()

        current_loss = loss.item()

        total_loss += current_loss
        global_step += 1

        # 6 step마다 학습 상태 출력
        if global_step % 6 == 0:
            print(
                f"Step {global_step:02d} | "
                f"Loss: {current_loss:.4f}"
            )

    average_loss = total_loss / len(empathetic_train)
    epoch_losses.append(average_loss)

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} completed | "
        f"Average Loss: {average_loss:.4f}\n"
    )

print("=== Training Completed ===")
print("Total steps:", global_step)
print("Epoch losses:", epoch_losses)

# ========================================
# 4. Google Drive에 LoRA 어댑터 저장
# ========================================

SAVE_DIR = Path(
    "/content/drive/MyDrive/"
    "lora-steering-experiment/adapters/empathetic"
)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(
    str(SAVE_DIR),
    selected_adapters=["empathetic"],
    safe_serialization=True
)

print("\n=== Adapter Saved ===")
print("Save directory:", SAVE_DIR)

# 실제 저장된 파일 목록 확인
for path in sorted(SAVE_DIR.rglob("*")):
    if path.is_file():
        print(
            path.relative_to(SAVE_DIR),
            f"({path.stat().st_size:,} bytes)"
        )

print("\nEmpathetic LoRA training and saving completed!")


## 7. 직언형 어댑터 생성

공감형 어댑터는 유지하되 비활성화하고, 동일한 기반 모델에 별도의 `direct` 어댑터를 만듭니다. 새 어댑터의 파라미터만 학습할 수 있는지 검증합니다.


In [ ]:
import torch

# 1. 현재 공감형 어댑터 확인
assert "empathetic" in model.peft_config

# 2. 기존 공감형과 동일한 LoRA 설정 사용
lora_config = model.peft_config["empathetic"]

# 3. 직언형 어댑터 생성
if "direct" not in model.peft_config:

    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    model.add_adapter(
        "direct",
        lora_config
    )

# 4. 직언형 어댑터 활성화
model.set_adapter("direct")

# 5. 생성 및 활성화 결과 확인
print("=== Available Adapters ===")
print(list(model.peft_config.keys()))

print("\n=== Active Adapter ===")
print(model.active_adapter)

# 6. 학습 가능한 파라미터 확인
trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

print("\n=== Trainable Parameters ===")
model.print_trainable_parameters()

# 7. 공감형 LoRA와 기본 모델이 학습 대상에서
# 제외되어 있는지 확인
assert "direct" in model.peft_config
assert model.active_adapter == "direct"

assert len(trainable_names) > 0

assert all(
    ".direct." in name
    for name in trainable_names
), "직언형 이외의 파라미터가 학습 가능 상태입니다."

print("\nDirect LoRA configuration: OK")


## 8. 직언형 LoRA 학습 및 Drive 저장

이 셀도 **한 번만 실행**하세요. 원본 실험의 epoch 평균 loss는 2.6980 → 2.3371 → 2.0128이었습니다. 두 데이터셋의 답변 길이·내용이 다르므로 서로 다른 loss 값으로 공감·직언 조향 성능을 비교할 수 없습니다.


In [ ]:
# ============================================================
# STEP 25: Direct LoRA Training and Saving
# ============================================================

import torch
import random
from pathlib import Path
from torch.optim import AdamW


# ============================================================
# 1. 학습 환경 확인
# ============================================================

# 모델이 존재하는지 확인
assert "model" in globals(), (
    "모델이 로드되지 않았습니다."
)

# 학습 데이터가 존재하는지 확인
assert "direct_train" in globals(), (
    "직언형 학습 데이터가 없습니다."
)

assert len(direct_train) == 12, (
    "직언형 학습 데이터 수가 12개가 아닙니다."
)

# 직언형 어댑터가 존재하는지 확인
assert "direct" in model.peft_config, (
    "직언형 LoRA 어댑터가 존재하지 않습니다."
)

# 활성화된 어댑터 확인
assert model.active_adapter == "direct", (
    "현재 활성화된 어댑터가 direct가 아닙니다."
)

# GPU 확인
assert torch.cuda.is_available(), (
    "CUDA GPU를 사용할 수 없습니다."
)

# 중복 학습 방지
assert not globals().get(
    "direct_training_completed", False
), "직언형 LoRA 학습이 이미 완료되었습니다."


# 직언형 LoRA만 학습 가능한 상태인지 확인
trainable_names = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

assert len(trainable_names) > 0

assert all(
    ".direct." in name
    for name in trainable_names
), (
    "직언형 LoRA 이외의 파라미터가 "
    "학습 가능한 상태입니다."
)


print("=== Training Environment ===")
print("Base model:", MODEL_ID)
print("Active adapter:", model.active_adapter)
print("Training samples:", len(direct_train))
print("GPU:", torch.cuda.get_device_name(0))

print(
    "Trainable parameters:",
    sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )
)


# ============================================================
# 2. 학습 설정
# ============================================================

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
RANDOM_SEED = 42

# 학습 데이터 순서의 재현성을 위한 난수 설정
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

# 직언형 LoRA 파라미터만 학습
trainable_params = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

# Optimizer
optimizer = AdamW(
    trainable_params,
    lr=LEARNING_RATE
)

# FP16 학습을 위한 Gradient Scaler
scaler = torch.amp.GradScaler("cuda")

# 학습 모드 활성화
model.train()


print("\n=== Training Configuration ===")
print("Epochs:", NUM_EPOCHS)
print("Batch size:", 1)
print("Learning rate:", LEARNING_RATE)
print("Optimizer: AdamW")
print("Precision: FP16")


# ============================================================
# 3. 직언형 LoRA 학습
# ============================================================

print("\n=== Direct LoRA Training ===")
print("Training started...\n")

global_step = 0
epoch_losses = []

for epoch in range(NUM_EPOCHS):

    # 매 Epoch마다 학습 데이터 순서 변경
    sample_indices = list(range(len(direct_train)))

    random.shuffle(sample_indices)

    total_loss = 0.0

    for idx in sample_indices:

        sample = direct_train[idx]

        # ----------------------------------------
        # 3-1. 학습 데이터 GPU 텐서 변환
        # ----------------------------------------

        batch = {
            key: torch.tensor(
                [value],
                dtype=torch.long,
                device="cuda"
            )
            for key, value in sample.items()
        }

        # 이전 단계의 Gradient 초기화
        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------
        # 3-2. Forward Pass
        # ----------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(**batch)

            loss = outputs.loss

        # Loss 값 확인
        if not torch.isfinite(loss).item():

            raise RuntimeError(
                f"Invalid loss at step "
                f"{global_step + 1}: "
                f"{loss.item()}"
            )

        # ----------------------------------------
        # 3-3. Backward Pass
        # ----------------------------------------

        scaler.scale(loss).backward()

        # ----------------------------------------
        # 3-4. Gradient Clipping
        # ----------------------------------------

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0
        )

        # ----------------------------------------
        # 3-5. LoRA 파라미터 업데이트
        # ----------------------------------------

        scaler.step(optimizer)

        scaler.update()

        # ----------------------------------------
        # 3-6. 학습 결과 기록
        # ----------------------------------------

        current_loss = loss.item()

        total_loss += current_loss

        global_step += 1

        # 6 step마다 진행 상황 출력
        if global_step % 6 == 0:

            print(
                f"Step {global_step:02d} | "
                f"Loss: {current_loss:.4f}"
            )

    # --------------------------------------------
    # Epoch 평균 Loss 계산
    # --------------------------------------------

    average_loss = total_loss / len(direct_train)

    epoch_losses.append(average_loss)

    print(
        f"\nEpoch {epoch + 1}/{NUM_EPOCHS} "
        f"completed | "
        f"Average Loss: {average_loss:.4f}\n"
    )


# ============================================================
# 4. 학습 완료 확인
# ============================================================

expected_steps = NUM_EPOCHS * len(direct_train)

assert global_step == expected_steps, (
    "예상 학습 Step 수와 실제 Step 수가 다릅니다."
)

print("=== Training Completed ===")

print("Total steps:", global_step)

print("Epoch losses:", epoch_losses)


# ============================================================
# 5. Google Drive에 직언형 LoRA 저장
# ============================================================

SAVE_DIR = Path(
    "/content/drive/MyDrive/"
    "lora-steering-experiment/"
    "adapters/direct"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# 직언형 LoRA 어댑터만 저장
model.save_pretrained(
    str(SAVE_DIR),
    selected_adapters=["direct"],
    safe_serialization=True
)


# ============================================================
# 6. 저장 결과 확인
# ============================================================

print("\n=== Adapter Saved ===")

print("Save directory:", SAVE_DIR)

saved_files = []

for path in sorted(SAVE_DIR.rglob("*")):

    if path.is_file():

        saved_files.append(path)

        print(
            path.relative_to(SAVE_DIR),
            f"({path.stat().st_size:,} bytes)"
        )


# 어댑터 가중치 파일이 실제로 존재하는지 확인
weight_files = [
    path
    for path in saved_files
    if path.name == "adapter_model.safetensors"
]

assert len(weight_files) == 1, (
    "직언형 LoRA 가중치 파일이 "
    "정상적으로 저장되지 않았습니다."
)

# 학습 완료 상태 기록
direct_training_completed = True

print(
    "\nDirect LoRA training "
    "and saving completed!"
)


## 9. 기본 모델과 단독 어댑터의 응답 비교

학습에 사용하지 않은 '승진 탈락' 질문을 공통 입력으로 사용하고, 기본 모델·공감형·직언형의 응답을 각각 생성합니다. 이 첫 비교는 `max_new_tokens=160`이므로 일부 응답이 잘릴 수 있습니다.


In [ ]:
# ============================================================
# STEP 26: Base / Empathetic / Direct Response Comparison
# ============================================================

import torch

# ------------------------------------------------------------
# 1. 실험 환경 확인
# ------------------------------------------------------------

assert "empathetic" in model.peft_config
assert "direct" in model.peft_config

assert next(model.parameters()).device.type == "cuda"

# 추론 모드
model.eval()

print("=== Available Adapters ===")
print(list(model.peft_config.keys()))


# ------------------------------------------------------------
# 2. 평가용 질문
# ------------------------------------------------------------

TEST_PROMPT = (
    "I was rejected for a promotion at work, "
    "and now I doubt my abilities. "
    "What should I do next?"
)

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Respond to the user's question naturally."
)

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": TEST_PROMPT
    }
]


# ------------------------------------------------------------
# 3. 모델 입력 생성
# ------------------------------------------------------------

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

device = next(model.parameters()).device

# Transformers 버전에 따른 반환 형식 처리
if isinstance(inputs, torch.Tensor):

    input_ids = inputs.to(device)

    attention_mask = torch.ones_like(input_ids)

else:

    input_ids = inputs["input_ids"].to(device)

    attention_mask = inputs["attention_mask"].to(device)


# ------------------------------------------------------------
# 4. 공통 응답 생성 함수
# ------------------------------------------------------------

def generate_response():

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 입력을 제외한 생성 결과만 추출
    generated_ids = outputs[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response.strip()


# ------------------------------------------------------------
# 5. 기본 모델 응답 생성
# ------------------------------------------------------------

print("\n=== Generating Base Model Response ===")

# 모든 LoRA를 일시적으로 비활성화
with model.disable_adapter():

    base_response = generate_response()


# ------------------------------------------------------------
# 6. 공감형 LoRA 응답 생성
# ------------------------------------------------------------

print("=== Generating Empathetic Response ===")

model.set_adapter("empathetic")

empathetic_response = generate_response()


# ------------------------------------------------------------
# 7. 직언형 LoRA 응답 생성
# ------------------------------------------------------------

print("=== Generating Direct Response ===")

model.set_adapter("direct")

direct_response = generate_response()


# ------------------------------------------------------------
# 8. 결과 출력
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST PROMPT")
print("=" * 60)

print(TEST_PROMPT)


print("\n" + "=" * 60)
print("1. BASE MODEL")
print("=" * 60)

print(base_response)


print("\n" + "=" * 60)
print("2. EMPATHETIC LORA")
print("=" * 60)

print(empathetic_response)


print("\n" + "=" * 60)
print("3. DIRECT LORA")
print("=" * 60)

print(direct_response)


print("\n" + "=" * 60)
print("COMPARISON COMPLETED")
print("=" * 60)


## 10. 50:50 가중치 변화량 조합

PEFT `combination_type="cat"`으로 두 LoRA 행렬을 이어 붙여 `0.5 ΔW_emp + 0.5 ΔW_direct`를 나타내는 새 어댑터를 만듭니다. A와 B 행렬을 따로 산술평균하는 방법과는 다릅니다. rank는 8+8=16입니다.


In [ ]:
# ============================================================
# STEP 27: 50:50 LoRA Weight Interpolation
# ============================================================

import torch

# 1. 기존 어댑터 확인
assert "empathetic" in model.peft_config
assert "direct" in model.peft_config

# 2. 조합 어댑터 이름
MIX_NAME = "mix_050"

# 3. 두 LoRA의 가중치 변화량을 50:50으로 조합
if MIX_NAME not in model.peft_config:

    model.add_weighted_adapter(
        adapters=[
            "empathetic",
            "direct"
        ],
        weights=[
            0.5,
            0.5
        ],
        adapter_name=MIX_NAME,
        combination_type="cat"
    )

# 4. 조합 어댑터 활성화
model.set_adapter(MIX_NAME)
model.eval()

# 5. 어댑터 상태 확인
print("=== Available Adapters ===")
print(list(model.peft_config.keys()))

print("\n=== Active Adapter ===")
print(model.active_adapter)

print("\n=== Combined Adapter Configuration ===")
print("Rank:", model.peft_config[MIX_NAME].r)
print("Alpha:", model.peft_config[MIX_NAME].lora_alpha)

# 6. 학습에 사용하지 않은 평가 질문
TEST_PROMPT = (
    "I was rejected for a promotion at work, "
    "and now I doubt my abilities. "
    "What should I do next?"
)

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Respond to the user's question naturally."
)

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": TEST_PROMPT
    }
]

# 7. 모델 입력 생성
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

device = next(model.parameters()).device

if isinstance(inputs, torch.Tensor):

    input_ids = inputs.to(device)
    attention_mask = torch.ones_like(input_ids)

else:

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

# 8. 조합 어댑터를 이용한 응답 생성
with torch.inference_mode():

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[
    0,
    input_ids.shape[-1]:
]

mix_response = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

# 9. 결과 출력
print("\n" + "=" * 60)
print("TEST PROMPT")
print("=" * 60)
print(TEST_PROMPT)

print("\n" + "=" * 60)
print("MIX 50:50 RESPONSE")
print("=" * 60)
print(mix_response)

print("\n" + "=" * 60)
print("INTERPOLATION COMPLETED")
print("=" * 60)


## 11. 다섯 가지 조합 비율 비교 및 JSON 저장

공감형 비율 0/25/50/75/100%의 응답을 생성하고 Google Drive에 `mixing_ratio_results.json`으로 저장합니다. 이 셀은 원본과 같이 `max_new_tokens=160`을 사용합니다.


In [ ]:
# ============================================================
# STEP 28: LoRA Mixing Ratio Comparison
# ============================================================

import torch
import json
from pathlib import Path

# ------------------------------------------------------------
# 1. 기존 어댑터와 실험 환경 확인
# ------------------------------------------------------------

assert "empathetic" in model.peft_config
assert "direct" in model.peft_config
assert "mix_050" in model.peft_config

assert torch.cuda.is_available()

# 추론 모드
model.eval()

print("=== Experiment Setup ===")
print("Base model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. 새로운 조합 어댑터 생성
# ------------------------------------------------------------

# 공감형 25% + 직언형 75%
if "mix_025" not in model.peft_config:

    model.add_weighted_adapter(
        adapters=["empathetic", "direct"],
        weights=[0.25, 0.75],
        adapter_name="mix_025",
        combination_type="cat"
    )


# 공감형 75% + 직언형 25%
if "mix_075" not in model.peft_config:

    model.add_weighted_adapter(
        adapters=["empathetic", "direct"],
        weights=[0.75, 0.25],
        adapter_name="mix_075",
        combination_type="cat"
    )


print("\n=== Available Adapters ===")
print(list(model.peft_config.keys()))


# ------------------------------------------------------------
# 3. 평가용 입력
# ------------------------------------------------------------

TEST_PROMPT = (
    "I was rejected for a promotion at work, "
    "and now I doubt my abilities. "
    "What should I do next?"
)

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Respond to the user's question naturally."
)

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": TEST_PROMPT
    }
]


# ------------------------------------------------------------
# 4. 입력 토큰 생성
# ------------------------------------------------------------

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

device = next(model.parameters()).device

if isinstance(inputs, torch.Tensor):

    input_ids = inputs.to(device)
    attention_mask = torch.ones_like(input_ids)

else:

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)


# ------------------------------------------------------------
# 5. 응답 생성 함수
# ------------------------------------------------------------

def generate_response():

    model.eval()

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return response.strip()


# ------------------------------------------------------------
# 6. 비율별 응답 생성
# ------------------------------------------------------------

# 공감형 비율, 어댑터 이름
mixing_conditions = [
    (0.00, "direct"),
    (0.25, "mix_025"),
    (0.50, "mix_050"),
    (0.75, "mix_075"),
    (1.00, "empathetic")
]

results = []

print("\n=== Mixing Ratio Experiment ===")

for ratio, adapter_name in mixing_conditions:

    model.set_adapter(adapter_name)

    response = generate_response()

    result = {
        "empathetic_ratio": ratio,
        "direct_ratio": 1.0 - ratio,
        "adapter": adapter_name,
        "prompt": TEST_PROMPT,
        "response": response
    }

    results.append(result)

    print("\n" + "=" * 60)

    print(
        f"Empathetic: {ratio * 100:.0f}% | "
        f"Direct: {(1.0 - ratio) * 100:.0f}%"
    )

    print("=" * 60)

    print(response)


# ------------------------------------------------------------
# 7. Google Drive에 결과 저장
# ------------------------------------------------------------

RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "lora-steering-experiment/results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_FILE = RESULT_DIR / "mixing_ratio_results.json"

with open(
    RESULT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model": MODEL_ID,
            "prompt": TEST_PROMPT,
            "generation": {
                "max_new_tokens": 160,
                "do_sample": False
            },
            "results": results
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 8. 실험 완료 확인
# ------------------------------------------------------------

assert len(results) == 5

assert RESULT_FILE.exists()

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETED")
print("=" * 60)

print("Total conditions:", len(results))
print("Results saved to:", RESULT_FILE)


## 12. 추가 세 맥락 비교 및 JSON 저장

서비스 장애, 가족의 해외 이주 비판, 두 가지 취업 제안에 관한 추가 평가 질문 3개 × 조합 비율 5개 = 15개 응답을 생성합니다. `max_new_tokens=256`을 적용합니다. **응답 길이 변화와 공감도 변화는 같지 않습니다.** 정량적 평가가 없으므로 이번 결과는 예비 관찰에 한정됩니다.


In [ ]:
# ============================================================
# STEP 29: Multiple-Context LoRA Evaluation
# ============================================================

import json
import torch
from pathlib import Path

# 1. 기존 어댑터 확인

mixing_conditions = [
    (0.00, "direct"),
    (0.25, "mix_025"),
    (0.50, "mix_050"),
    (0.75, "mix_075"),
    (1.00, "empathetic"),
]

for ratio, adapter_name in mixing_conditions:
    assert adapter_name in model.peft_config, (
        f"Adapter missing: {adapter_name}"
    )

assert torch.cuda.is_available()

model.eval()


# 2. 새로운 평가 질문

test_prompts = [
    (
        "I introduced a bug that caused our service to "
        "go down. I feel guilty about letting my team "
        "down. What should I do first?"
    ),
    (
        "A family member criticized my decision to "
        "move abroad, and I feel hurt. "
        "How should I talk to them?"
    ),
    (
        "I have two job offers, but I am afraid of "
        "making the wrong decision. "
        "How should I choose between them?"
    ),
]

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Respond to the user's question naturally."
)


# 3. 공통 응답 생성 함수

def generate_test_response(prompt, adapter_name):

    model.set_adapter(adapter_name)
    model.eval()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    device = next(model.parameters()).device

    if isinstance(inputs, torch.Tensor):
        input_ids = inputs.to(device)
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)

    max_new_tokens = 256

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    # 최대 토큰 수에 도달했다면 응답이
    # 중간에 잘렸을 가능성이 있음
    length_limit_reached = (
        len(generated_ids) >= max_new_tokens
    )

    return {
        "response": response,
        "generated_tokens": len(generated_ids),
        "length_limit_reached": length_limit_reached
    }


# 4. 맥락별 / 조합 비율별 응답 생성

results = []

for prompt_index, prompt in enumerate(
    test_prompts,
    start=1
):

    print("\n" + "=" * 65)
    print(f"TEST PROMPT {prompt_index}")
    print("=" * 65)
    print(prompt)

    for ratio, adapter_name in mixing_conditions:

        result = generate_test_response(
            prompt,
            adapter_name
        )

        record = {
            "prompt_index": prompt_index,
            "prompt": prompt,
            "empathetic_ratio": ratio,
            "direct_ratio": 1.0 - ratio,
            "adapter": adapter_name,
            **result
        }

        results.append(record)

        print("\n" + "-" * 65)

        print(
            f"Empathetic: {ratio * 100:.0f}% | "
            f"Direct: {(1.0 - ratio) * 100:.0f}%"
        )

        print("-" * 65)

        print(result["response"])

        print(
            "\nGenerated tokens:",
            result["generated_tokens"]
        )

        print(
            "Length limit reached:",
            result["length_limit_reached"]
        )


# 5. Google Drive에 결과 저장

RESULT_DIR = Path(
    "/content/drive/MyDrive/"
    "lora-steering-experiment/results"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_FILE = (
    RESULT_DIR / "multiple_context_results.json"
)

with open(
    RESULT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "model": MODEL_ID,
            "generation": {
                "max_new_tokens": 256,
                "do_sample": False
            },
            "results": results
        },
        f,
        ensure_ascii=False,
        indent=2
    )


# 6. 완료 확인

assert len(results) == 15
assert RESULT_FILE.exists()

print("\n" + "=" * 65)
print("EXPERIMENT COMPLETED")
print("=" * 65)

print("Evaluation prompts:", len(test_prompts))
print("Mixing conditions:", len(mixing_conditions))
print("Total responses:", len(results))
print("Results saved to:", RESULT_FILE)


## 결과 파일과 실험 한계

- [최초 평가 질문 결과 (5개 응답)](https://github.com/mongkoon/lora-steering-experiment/blob/main/results/mixing_ratio_results.json)
- [다중 맥락 결과 (15개 응답)](https://github.com/mongkoon/lora-steering-experiment/blob/main/results/multiple_context_results.json)

본 실험에서 **구현한 것**은 서로 다른 스타일의 LoRA 두 개를 학습하고, 해당 weight delta를 서로 다른 비율로 조합하여 생성 응답을 비교한 과정입니다. **구현하지 않은 것**은 입력 맥락으로 최적 비율을 자동 산출하는 매핑 모델, 정렬 성능의 객관적 평가, layer 선택, 실제 온디바이스 벤치마크입니다.

> 출처·기록 구분: GitHub `results/`의 기존 JSON은 **이전에 실행해 저장한 관찰 결과**이고, 이 정리본 노트북은 **오류·재연결 과정을 제거한 재현 절차**입니다. 이 파일의 코드 셀은 새로 실행한 출력이 없으므로 JSON 응답이 동일하게 재현되었다고 해석하면 안 됩니다.
